# Distributed Acoustic Sensing (DAS): data, noise and machine-learning denoising

Intelligent Earth CDT · Natural Hazards week · Seismology practical

This practical follows on from the morning's talk on DAS and DAS-N2N. You will work with DAS recordings from three settings:

| Site | Fibre deployment | Signals |
|---|---|---|
| Rutford Ice Stream, West Antarctica | 1 km loop on the ice surface | icequakes at the ice-stream bed |
| Utah FORGE geothermal site, USA | ~1 km in a vertical borehole | a microearthquake |
| Ocean Observatories Initiative, offshore Oregon | seafloor telecommunications cable | whale calls and ocean noise |

### Using this notebook
Work through the cells in order. Some cells are marked:

* ✏️ a value for you to fill in: replace `None` with a number (for example, `fmin = None` becomes `fmin = 10`) and run the cell. These cells will stop with a message until they are filled in.
* 🎚️ sliders and drop-down menus to explore the data.

The questions are there to think about or discuss with the person next to you; there is no need to write the answers down. Answers, including the values for the ✏️ cells, are in collapsible sections after each set of questions (click ▸ Answers to expand); try the questions first.

### Suggested timing (about 1.5–2 hours)
| Part | Time |
|---|---|
| 1. The Rutford data | 10 min |
| 2. Noise characteristics | 15 min |
| 3. Conventional filters | 15 min |
| 4. DAS-N2N | 20 min |
| 5. DAS and geophones: the missing P wave | 15 min |
| 6. Choose one: A. borehole fibre (FORGE) or B. seafloor cable (OOI) | 20–30 min |
| 7. Optional: train a Noise2Noise model | — |

If you finish early, move on to the PhaseNet or Bayesian tomography practicals.

Before starting, select *Runtime → Change runtime type → T4 GPU*. The notebook also runs on a CPU, but Parts 6B and 7 are considerably faster on a GPU.

## 0. Setup

In [ ]:
#@title Run this cell first: installs packages and fetches the helper code (~1 min)
import os, importlib.util, urllib.request
REPO = "sachalapins/ie-cdt-das-practical-2026"   # holds das_utils.py; the data are under its Releases
os.environ.setdefault("DAS_PRACTICAL_DATA_URL", f"https://github.com/{REPO}/releases/download/v1.0/")

# Install ObsPy and DAS-N2N if they are missing (always the case on a fresh Colab runtime)
if any(importlib.util.find_spec(pkg) is None for pkg in ("obspy", "dasn2n")):
    %pip install -q obspy git+https://github.com/sachalapins/DAS-N2N-torch.git

# Fetch the practical's helper module (data files are downloaded automatically when first used)
if not os.path.exists("das_utils.py"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/{REPO}/main/das_utils.py", "das_utils.py")

import numpy as np
import matplotlib.pyplot as plt
import das_utils as du          # plotting, filtering and scoring functions written for this practical

plt.rcParams.update({"figure.dpi": 90, "font.size": 10})
print("Ready.")

---
## Part 1. The Rutford data

### What DAS measures
An interrogator sends short laser pulses into an optical fibre and records the light that is Rayleigh-backscattered from small inhomogeneities in the glass. Deformation of the fibre changes the optical phase of the backscattered light. The phase difference between two points separated by the gauge length, $L_g$ (10 m here), is proportional to the strain of the fibre between them, and this is measured every metre or so along the cable. Each measurement point is referred to as a channel.

The interrogators used here output strain rate along the fibre axis:

$$\dot\varepsilon_{xx} = \frac{\partial v_x}{\partial x},$$

where $x$ is the along-fibre direction and $v_x$ is the component of ground (particle) velocity in that direction. A DAS channel is therefore sensitive to only one component of the wavefield, unlike a three-component seismometer, which records ground velocity in the vertical and two horizontal directions. This becomes important in Part 5.

### Rutford Ice Stream
In January 2020, a 1 km fibre-optic cable was laid out on the surface of Rutford Ice Stream, West Antarctica, in a triangle with sides of approximately 330 m. The ice is about 2 km thick and flows at hundreds of metres per year; stick–slip motion at the bed generates large numbers of small icequakes. The interrogator was powered by a diesel generator located beside it.

In [ ]:
rec = du.load_das("rutford_event1.npz")
rec

`rec.data` is a 2-D array with shape (time samples, channels); `rec.fs` is the sampling rate (Hz) and `rec.dx` the channel spacing (m).

DAS produces large volumes of data. Run the cell below to see how much.

In [ ]:
bytes_per_second = rec.nch * rec.fs * 4     # channels × samples per second × 4 bytes per value
per_day_GB = bytes_per_second * 86400 / 1e9
print(f"This 1 km cable:  {bytes_per_second/1e6:.1f} MB/s  =  {per_day_GB:.0f} GB/day  =  {per_day_GB*365/1e3:.0f} TB/year")

# A typical seafloor telecommunications cable: 50 km of fibre, a channel every 2 m, sampled at 200 Hz
offshore = 50_000 / 2 * 200 * 4
print(f"50 km seafloor cable: {offshore/1e6:.0f} MB/s  =  {offshore*86400/1e12:.1f} TB/day")

**Q1.1** What does this imply for how DAS data can be processed and analysed, and for the computational cost of any denoising method?

### Viewing the data
DAS data are usually displayed as an image, with distance along the fibre on the horizontal axis, time increasing downwards, and colour showing the sign and amplitude of the strain rate. This record contains an icequake.

In [ ]:
du.plot_record(rec);

🎚️ Adjust the colour scale and select individual channels. A lower percentile reveals weaker features but saturates the strongest ones. Can you identify the icequake in the image or in a single channel?

In [ ]:
#@title 🎚️ Explore the raw data { run: "auto" }
colour_percentile = 99   #@param {type:"slider", min:50, max:100, step:0.5}
channel = 450            #@param {type:"slider", min:0, max:985, step:1}

fig, axs = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})
du.plot_record(rec, perc=colour_percentile, ax=axs[0])
axs[0].axvline(rec.x[channel], color="k", lw=1, ls="--")
axs[1].plot(rec.data[:, channel], rec.t, "k", lw=0.5)
axs[1].set_ylim(rec.t[-1], 0); axs[1].set_xlabel("strain rate"); axs[1].set_title(f"channel {channel}")
plt.tight_layout()

<details><summary>▸ Answers</summary>

Q1.1: roughly 340 GB/day for this short cable and ~1.7 TB/day for a long seafloor cable. Data at this volume cannot be inspected or labelled manually, so processing must be automated and faster than real time. This favours computationally cheap methods and self-supervised training.
In the raw data the icequake is not visible: the incoherent optical noise exceeds the signal amplitude. The visible structures are noisy individual channels (vertical stripes) and generator noise at both ends of the cable.

</details>

---
## Part 2. Noise characteristics

Before attempting to remove noise, it is useful to characterise it. Two broad sources contribute here:

* instrument (optical) noise from the laser and photodetector, which is similar along the whole fibre;
* environmental noise from sources near the cable, such as the generator, which is strongest close to its source.

The amplitude spectrum shows how the energy is distributed with frequency. The cell below averages spectra over three groups of channels.

**Q2.1** Before running it: would you expect the spectra from the start, middle and end of the cable to be the same?

In [ ]:
du.plot_spectra(rec, {"channels 0–100 (start)": slice(0, 100),
                      "channels 400–500 (middle)": slice(400, 500),
                      "channels 886–986 (end)": slice(886, 986)});

✏️ Read off the frequencies of the two lowest sharp peaks. To zoom in, you can add `fmax=150` to the `plot_spectra` call above.

In [ ]:
peak1_hz = None    # ✏️
peak2_hz = None    # ✏️

du.check_range("peak1_hz", peak1_hz, 30, 36, "Hz")
du.check_range("peak2_hz", peak2_hz, 63, 69, "Hz")

<details><summary>▸ Answers</summary>

Peaks at ≈33 Hz and ≈66 Hz (`peak1_hz = 33`, `peak2_hz = 66`). They are produced by the diesel generator that powered the interrogator: its engine runs at a near-constant rate, producing a fundamental at about 33 Hz and harmonics at multiples of it. The resulting vibrations propagate through the near-surface snow and firn as surface waves.

</details>

**Q2.2** The generator signal is recorded at both ends of the cable but not in the middle. What does this indicate about the cable layout, given that the cable forms a closed triangle?

**Q2.3** Away from the generator, the noise amplitude increases with frequency up to about 200 Hz. The icequake signals are mostly at 10–100 Hz. What does this suggest about a suitable filter?

<details><summary>▸ Answers</summary>

Q2.1: no. Channels 0–100 show many additional lines: harmonics of the ≈33.6 Hz generator frequency up to about the ninth (≈67, 101, 134, 168, 235, 268, 302 Hz), a second, slightly offset harmonic series (≈103, 137, 172 Hz, i.e. multiples of ≈34.3 Hz, suggesting a second rotating machine), and several unrelated lines (e.g. ≈82, 329, 412, 463 and 497 Hz). The fibre between the interrogator and channel 0 (not included in this file) records the same lines more strongly, so they originate from equipment close to the interrogator. High-frequency vibration attenuates rapidly in snow and firn, so these lines are recorded only near their source; the lower generator frequencies (33 and 66 Hz) propagate further and are also recorded at the far end of the cable.
Q2.2: the loop starts and ends at the interrogator, which is beside the generator, so channels 0 and 985 are both close to the noise source.
Q2.3: a band-pass filter with an upper corner around 100 Hz removes much of the noise energy while retaining most of the signal (Part 3).

</details>

### Optional: the frequency–wavenumber (f–k) spectrum
A 2-D Fourier transform of a record section gives its amplitude as a function of frequency $f$ and wavenumber $k$ (cycles per metre along the fibre). A wave with apparent velocity $v$ along the fibre plots on the straight line $f = v\,k$ through the origin, so faster waves plot on steeper lines.

The cell below compares the f–k spectrum of a 0.6 s window containing the icequake with that of a 0.6 s window containing only noise. Both are shown for wavenumbers up to 0.1 cycles/m.

**Q2.4** Which features appear only in the window containing the icequake? What apparent velocities do they correspond to? Which features appear in both windows?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
du.plot_fk(rec, data=rec.data[700:1300], ax=axs[0], fmax=200, kmax=0.1, title="0.7–1.3 s (icequake)")
du.plot_fk(rec, data=rec.data[1500:2100], ax=axs[1], fmax=200, kmax=0.1, title="1.5–2.1 s (noise only)")
axs[1].set_ylabel("")
plt.tight_layout()

<details><summary>▸ Answers</summary>

The icequake window contains two straight lines through the origin, one for each of the two sides of the triangle on which the S wave is recorded. For example, a line passing through f ≈ 150 Hz at k ≈ 0.035 cycles/m corresponds to v = f/k ≈ 4.3 km/s; the steeper line corresponds to ≈6 km/s. These are apparent velocities along the cable, considerably higher than the S-wave velocity in ice (≈1.9 km/s), which is explained in Part 5. The narrow horizontal lines at ≈33 and ≈67 Hz, at small non-zero wavenumbers, are the generator signal and appear in both windows; the broad background is the incoherent instrument noise.

</details>

---
## Part 3. Conventional filters

### 3.1 Band-pass filtering
A band-pass filter retains frequencies between `fmin` and `fmax` and attenuates the rest. It is applied to each channel's time series independently.

✏️ Choose a pass band using the spectra from Part 2, then 🎚️ try alternatives. What happens if the band includes 33 Hz, or if `fmax` is 400 Hz?

In [ ]:
fmin = None   # ✏️ Hz
fmax = None   # ✏️ Hz

du.require(fmin, fmax)
bp = du.bandpass(rec.data, rec.fs, fmin, fmax)
du.compare({"raw": rec.data, f"band-pass {fmin}–{fmax} Hz": bp, "removed (raw − filtered)": rec.data - bp},
           rec, same_scale=False);

The icequake is now visible as a shear (S) wave arriving at slightly different times along the cable.

**Q3.1** Why does the arrival consist of straight segments separated by abrupt changes in slope?

### 3.2 Quantifying performance
To compare methods we use a signal-to-noise ratio (SNR) in decibels,

$$\mathrm{SNR} = 20\log_{10}\frac{S}{N}, \qquad N = \mathrm{RMS}(\text{noise window}), \qquad S = \sqrt{\mathrm{RMS}(\text{signal window})^2 - N^2},$$

where the signal window encloses the clearest part of the S-wave arrival and the noise window is a quiet interval later in the record. Subtracting $N^2$ removes the contribution of noise within the signal window. An increase of 6 dB corresponds to a doubling of the amplitude ratio. Because the arrival occupies only a small part of the signal window, absolute values are low; the comparison between methods is what matters.

A denoising method should also preserve signal amplitude, so we additionally record the amplitude retained relative to a 10–100 Hz band-pass filter (the benchmark used by Lapins et al., 2024), both for the main S arrival and for the weaker, earlier arrival at ≈0.85 s.

In [ ]:
SIGNAL_BOX = ((0.94, 1.08), (150, 650))     # ((t_start, t_end) in s, (first channel, last channel))
WEAK_BOX   = ((0.81, 0.89), (150, 650))
NOISE_BOX  = ((1.50, 2.50), (150, 650))

reference = du.bandpass(rec.data, rec.fs, 10, 100)
board = du.Scoreboard(rec, SIGNAL_BOX, NOISE_BOX, reference=reference, weak_box=WEAK_BOX)
times = {}

ax = du.plot_record(rec, reference, title="10–100 Hz band-pass with the scoring windows")
du.draw_boxes(ax, rec, SIGNAL_BOX, NOISE_BOX, WEAK_BOX)

board.add("raw", rec.data, amplitude=False)
with du.timer("band-pass (yours)", times):
    bp = du.bandpass(rec.data, rec.fs, fmin, fmax)
board.add(f"band-pass {fmin}-{fmax} Hz", bp, times["band-pass (yours)"])
board.show()

🎚️ You can return to 3.1, change the pass band and re-run both cells to add it to the table. Does the highest SNR always correspond to the best-looking result?

### 3.3 Wiener filtering
A band-pass filter operates only along the time axis. A two-dimensional Wiener filter estimates the local mean and variance in a small window spanning both time and channels, and smooths more strongly where the local variance is close to the estimated noise variance. Here it is applied to your band-passed data.

In [ ]:
#@title 🎚️ Wiener filter window { run: "auto" }
window_time_samples = 7   #@param {type:"slider", min:3, max:31, step:2}
window_channels = 7       #@param {type:"slider", min:1, max:31, step:2}

with du.timer("Wiener", times):
    wf = du.wiener(bp, (window_time_samples, window_channels))
board.add(f"Wiener {window_time_samples}x{window_channels}", wf, times["Wiener"])
du.compare({"band-passed": bp, "band-pass + Wiener": wf, "removed by Wiener": bp - wf}, rec, same_scale=False)
board.show()

**Q3.2** How do the S-wave arrival and the noisy channels change as the window size increases? Is any signal visible in the "removed" panel?

**Q3.3** Try a 21 × 21 window. Its SNR is higher than that of any other method in this notebook, including DAS-N2N in Part 4. Compare the two amplitude columns and look at the weaker arrival at ≈0.85 s. Is the higher SNR a better result?

<details><summary>▸ Answers</summary>

A pass band of about 10–100 Hz works well (`fmin = 10`, `fmax = 100`). Including 33 Hz leaves the generator noise at the cable ends; raising `fmax` to 400 Hz admits much more of the high-frequency noise.
Q3.1: the cable is a triangle, so the distance from the source to the fibre changes at a different rate along each side; the abrupt changes in slope occur at the corners (at ~330 m and ~660 m).
Q3.2: larger windows increase the SNR but progressively smear and attenuate the S wave. Noisy channels are spread laterally rather than removed. Coherent signal in the removed panel indicates signal leakage, which should be checked for any denoising method.
Q3.3: no. The S wave is almost flat across the cable (apparent velocity ≈4–6 km/s), so averaging over 21 channels and 21 ms acts much like stacking along the arrival, and the noise window is smoothed almost to zero. The SNR of the main arrival therefore rises to ≈16 dB, but only ≈60% of its amplitude is retained, the waveform is smoothed, and only ≈25% of the weaker arrival's amplitude survives (DAS-N2N: ≈70% and ≈55%). A single SNR value measured on the strongest arrival rewards heavy smoothing; amplitude retention, waveform shape and the preservation of weak or more steeply dipping signals must also be considered. The heavy smoothing would also perform poorly for slower waves with steeper moveout across the channels.

</details>

---
## Part 4. Machine-learning denoising with DAS-N2N

### Noise2Noise
Supervised denoising normally requires pairs of noisy and clean examples, which do not exist for DAS data. Noise2Noise (Lehtinen et al., 2018) avoids this requirement. Consider two recordings of the same signal $s$ with independent, zero-mean noise:

$$y_1 = s + n_1, \qquad y_2 = s + n_2 .$$

A network $f$ is trained to predict $y_2$ from $y_1$ by minimising the expected squared error. Because $n_2$ is independent of $y_1$ and has zero mean, the cross term vanishes:

$$\mathbb{E}\,\|f(y_1) - y_2\|^2 = \mathbb{E}\,\|f(y_1) - s\|^2 + \mathbb{E}\,\|n_2\|^2 .$$

The second term does not depend on $f$, so minimising the loss against noisy targets has the same minimiser as minimising it against clean targets. The noise in the target cannot be predicted from the input, so the loss is minimised by mapping it to its expected value, which is zero.

### Training pairs at Rutford
The Rutford cable contains two fibres that are spliced together at the far end of the loop, so each point on the ice is recorded twice, by physically separate fibres. These recordings share the same signal but have different realisations of the instrument noise. DAS-N2N was trained on such pairs (Lapins et al., 2024); it is a small U-Net with about 47,000 parameters that operates on 128-sample × 96-channel patches.

The data file includes both fibres, aligned channel by channel:

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
f1_bp = du.bandpass(rec.data, rec.fs, 10, 100)
f2_bp = du.bandpass(rec.fibre2, rec.fs, 10, 100)
du.plot_record(rec, f1_bp, ax=axs[0], title="fibre 1 (10–100 Hz)", tlim=(0.6, 1.4))
du.plot_record(rec, f2_bp, ax=axs[1], title="fibre 2 (10–100 Hz)", tlim=(0.6, 1.4))
du.plot_record(rec, (f1_bp + f2_bp) / 2, ax=axs[2], title="mean of the two fibres", tlim=(0.6, 1.4))
plt.tight_layout()

### 4.1 Testing the independence assumption
The cell below computes the correlation coefficient between the noise on the two fibres at the same location, using a time window with no icequake. ✏️ Complete the second line to compute the correlation between neighbouring channels (`c` and `c + 1`) on the same fibre.

In [ ]:
quiet = slice(1500, 3500)       # samples from 1.5 to 3.5 s, after the icequake
c = 450
f1 = du.bandpass(rec.data, rec.fs, 10, 400)[quiet]
f2 = du.bandpass(rec.fibre2, rec.fs, 10, 400)[quiet]

corr_fibres = np.corrcoef(f1[:, c], f2[:, c])[0, 1]
corr_neighbours = None   # ✏️ as above, but for f1[:, c] and f1[:, c + 1]

du.require(corr_neighbours)
print(f"noise correlation, fibre 1 vs fibre 2 at the same location: {corr_fibres:+.2f}")
print(f"noise correlation, neighbouring channels on one fibre:       {corr_neighbours:+.2f}")

**Q4.1** Why would neighbouring channels on a single fibre not be suitable as Noise2Noise training pairs? (Part 7 tests this directly.)

<details><summary>▸ Answers</summary>

`corr_neighbours = np.corrcoef(f1[:, c], f1[:, c + 1])[0, 1]`

The noise on the two fibres is uncorrelated (≈0), as Noise2Noise requires. Noise on neighbouring channels of the same fibre is very strongly correlated (≈0.9 or more in this band): with a 10 m gauge length and 1 m channel spacing, neighbouring channels largely measure the same section of fibre. A network trained on such pairs could predict much of the target noise from the input, and the loss would be minimised by reproducing the noise rather than suppressing it.

</details>

### 4.2 Applying the pre-trained model
`dasn2n_denoise` normalises each channel, divides the data into overlapping 128 × 96 patches, applies the network to each patch, and reassembles the output. The panels below share a colour scale of ±50, so amplitudes can be compared directly, and show the 1 s around the icequake.

In [ ]:
model = du.load_dasn2n()

with du.timer("DAS-N2N", times):
    den = du.dasn2n_denoise(model, rec.data)

board.add("DAS-N2N", den, times["DAS-N2N"])
du.compare({"raw": rec.data, "10–100 Hz band-pass": reference, "DAS-N2N": den,
            "removed by DAS-N2N": rec.data - den}, rec, vmax=50, tlim=(0.5, 1.5))
board.show()

**Q4.2**
* How does DAS-N2N compare with the conventional filters in SNR, retained signal amplitude and run time?
* Is the "removed" panel entirely noise?
* Which method shows the weaker, earlier arrival (≈0.85 s) most clearly?
* Using the data rates from Part 1, could DAS-N2N process the 1 km cable in real time on this hardware? The 50 km cable?

<details><summary>▸ Answers</summary>

Typical values: raw ≈ −15 dB; 10–100 Hz band-pass ≈ 0 dB; band-pass + 7×7 Wiener ≈ +4 dB (≈0.8 of the band-pass amplitude); DAS-N2N ≈ +9 dB (≈0.65–0.7 of the amplitude). Lapins et al. (2024) also report that DAS-N2N and Wiener outputs are ≈4/5 of the band-pass amplitude, and a faint trace of the S wave is visible in the removed panel.
On a 2-core Colab CPU, DAS-N2N takes ≈5 s for this 3.5 s record, i.e. roughly real time. On a GPU it processes 30 s × 985 channels in under a second (Lapins et al., 2024). A long seafloor cable would require GPU processing.

</details>

### 4.3 Data outside the training period
The model was trained on data recorded between 01:00 and 03:00 UTC on 17 January 2020. Event 1 (01:30) lies within this period; event 2 (04:42) was recorded afterwards and is therefore independent test data.

**Q4.3** Would you expect the model to perform as well on event 2? After running the cell, count the number of separate S-wave arrivals.

In [ ]:
rec2 = du.load_das("rutford_event2.npz")
den2 = du.dasn2n_denoise(model, rec2.data)
du.compare({"event 2 raw": rec2.data, "10–100 Hz band-pass": du.bandpass(rec2.data, rec2.fs, 10, 100),
            "DAS-N2N": den2}, rec2, vmax=50, tlim=(0.8, 2.4));

<details><summary>▸ Answers</summary>

Three S arrivals, at ≈1.3 s, ≈1.6 s and a weaker one at ≈1.8–1.9 s (clearest between ~300 and 500 m); the geophones (`du.load_geophones("ev2")`) record corresponding arrivals. Performance is comparable to event 1, as expected: the noise characteristics are unchanged, and the network has learned to suppress spatially and temporally incoherent noise rather than to reproduce particular events.

</details>

---
## Part 5. DAS and geophones: the missing P wave

Three-component geophones were deployed near the DAS cable, recording ground velocity on vertical (`GHZ`) and two horizontal (`GH1`, north; `GH2`, east) components.

An icequake radiates compressional (P) waves, with particle motion parallel to the propagation direction (≈3.8 km/s in ice), and shear (S) waves, with particle motion perpendicular to it (≈1.9 km/s). The icequakes occur at the bed, about 2 km beneath the cable.

**Q5.1** Before plotting: which geophone components would you expect to record the P wave most clearly, and which the S wave? Would you expect the DAS cable to record the P wave?

The geophone data are stored in digital counts. `remove_response` uses each instrument's calibration to convert them to ground velocity (m/s).

In [ ]:
geo, stations = du.load_geophones("ev1")
geo.remove_response(stations, output="VEL", pre_filt=[2, 4, 400, 450])   # counts -> m/s
geo

In [ ]:
#@title 🎚️ Geophone (top) and DAS (bottom) recordings on a common time axis { run: "auto" }
station = "A000"         #@param ["A000", "AS22", "R102", "R103", "R104", "R201", "R202", "R203"]
geo_fmin = 40            #@param {type:"slider", min:2, max:100, step:1}
geo_fmax = 300           #@param {type:"slider", min:50, max:400, step:10}
das_channel = 450        #@param {type:"slider", min:0, max:985, step:5}

from obspy import UTCDateTime
t0 = UTCDateTime(rec.starttime)
g = geo.select(station=station).copy().filter("bandpass", freqmin=geo_fmin, freqmax=geo_fmax, zerophase=True)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
for i, (cha, col) in enumerate([("GHZ", "k"), ("GH1", "tab:red"), ("GH2", "tab:blue")]):
    tr = g.select(channel=cha)[0]
    axs[0].plot(tr.times() + (tr.stats.starttime - t0), tr.data * 1e6 - i * 0.6, col, lw=0.7, label=cha)
axs[0].set_ylabel("ground velocity (µm/s), offset")
axs[0].legend(loc="upper right"); axs[0].set_title(f"geophone {station}, {geo_fmin}–{geo_fmax} Hz")
axs[1].plot(rec.t, den[:, das_channel], "g", lw=0.7)
axs[1].set_ylabel("strain rate"); axs[1].set_title(f"DAS-N2N, channel {das_channel}")
axs[1].set_xlabel(f"time (s) after {rec.starttime[:22]}")
axs[1].set_xlim(0, rec.t[-1])
plt.tight_layout()

✏️ Read off the P and S arrival times at a geophone and mark them on the DAS record section.

In [ ]:
p_time = None   # ✏️ seconds after the start of the record
s_time = None   # ✏️

du.require(p_time, s_time)
ax = du.plot_record(rec, den, vmax=50, title="DAS-N2N with the geophone arrival times", tlim=(0, 1.6))
ax.axhline(p_time, color="k", ls="--", lw=1.5, label="P (geophone)")
ax.axhline(s_time, color="g", ls="--", lw=1.5, label="S (geophone)")
ax.legend();

<details><summary>▸ Answers</summary>

For example, at A000: `p_time = 0.33`, `s_time = 0.92`.

At A000 (40–300 Hz) the vertical component shows a clear P arrival at ≈0.33 s (and a smaller one at ≈0.17 s), and the horizontal components show S arrivals at ≈0.77 s and ≈0.92 s, consistent with two icequakes and with the weak and strong S arrivals on the DAS. Arrival times differ by up to ≈0.1 s between stations. The DAS shows the S waves but nothing at the P arrival time.
Station R102 is dominated by the 33 Hz generator signal; a lower corner above ~40 Hz or `du.notch()` reduces it. Its S timing matches channel 0, so it is probably the geophone closest to the interrogator end of the loop.

</details>

### 5.1 Why is the P wave not recorded?
The upper ~100 m of the ice sheet is firn, compacted snow whose seismic velocity decreases towards the surface (Smith, 1997). By Snell's law, rays travelling upwards through a layer of decreasing velocity are refracted towards the vertical, so both P and S waves reach the surface at close to vertical incidence (Hudson et al., 2021).

For a near-vertical P wave, the particle motion is close to vertical, i.e. perpendicular to the horizontal fibre, and produces almost no strain along the fibre axis. For a near-vertical S wave, the particle motion is close to horizontal and therefore has a substantial component along the fibre. Consequently the S waves are recorded on the DAS and the P waves are not, whereas the three-component geophones record both. On glaciers without a firn layer, surface DAS deployments do record P waves (Walter et al., 2020).

The near-vertical arrival can be checked from the data. A wavefront arriving close to vertically reaches all points along the cable at almost the same time, so its apparent velocity along the cable is much higher than its true velocity.

✏️ Using the zoomed plot below, read off the S-wave arrival time at the two dashed channels, on the straight segment between ≈340 m and ≈600 m.

In [ ]:
#@title 🎚️ Zoomed view for picking { run: "auto" }
channel_a = 340   #@param {type:"slider", min:0, max:985, step:5}
channel_b = 600   #@param {type:"slider", min:0, max:985, step:5}
ax = du.plot_record(rec, den, tlim=(0.85, 1.1), title="S-wave arrival at the two dashed channels")
for ch in (channel_a, channel_b):
    ax.axvline(rec.x[ch], color="k", ls="--")
ax.grid(alpha=0.5); ax.minorticks_on(); ax.grid(which="minor", alpha=0.2)

In [ ]:
t_a = None   # ✏️ S arrival time (s) at channel_a
t_b = None   # ✏️ S arrival time (s) at channel_b

du.require(t_a, t_b)
v_app = abs(rec.x[channel_b] - rec.x[channel_a]) / abs(t_b - t_a)
print(f"apparent velocity along the cable ≈ {v_app:.0f} m/s  (S-wave velocity in ice ≈ 1900 m/s)")

**Q5.2**
* How does the apparent velocity compare with the S-wave velocity, and what does this indicate about the direction of arrival?
* The S wave is much weaker on the third side of the triangle (≈660–986 m). Why might one orientation of fibre record the wave much more weakly than the others?
* What are the relative advantages of a DAS cable and a small number of three-component seismometers?
* In Part 6A the fibre is installed vertically in a borehole. What would you expect to change?

<details><summary>▸ Answers</summary>

Picks of `t_a ≈ 0.978` (340 m) and `t_b ≈ 1.020` (600 m) give an apparent velocity of ≈6.2 km/s.

* The apparent velocity is ≈6 km/s, about three times the S-wave velocity, so the wavefront is close to horizontal as it reaches the cable, i.e. the waves arrive close to vertical incidence. (For arrival in the plane of the cable, sin i = v / v_app gives i ≲ 18°; Hudson et al. (2021) estimate ~22–25° from shear-wave splitting analysis and note that the true inclination may be steeper.)
* DAS measures only the along-fibre strain. On the third side, the horizontal S-wave particle motion is closer to perpendicular to the fibre, so the along-fibre strain is small. This directional sensitivity (approximately cos² of the angle between fibre and particle motion for a plane wave) is often described as broadside insensitivity. Hudson et al. (2021) use the different orientations of the triangle to recover multi-component information.
* DAS provides dense spatial sampling (a channel every metre over long distances, including locations difficult to instrument) but records one strain component, with direction-dependent sensitivity and higher self-noise. Seismometers record the full three-component motion at a limited number of points. The two are complementary.
* A vertical fibre is aligned with near-vertical P-wave particle motion, so P waves should be recorded well (Part 6A).

</details>

---
## Part 6A. A borehole fibre at Utah FORGE

FORGE is a field laboratory for enhanced geothermal systems in Utah. Here the fibre is cemented into a vertical well (78-32). Distance along the fibre is measured from the wellhead, so for $x > 0$ it approximates depth; channels with $x < 0$ are fibre at the surface. The record contains a microearthquake beneath the well.

**Q6A.1** With a vertical fibre and waves arriving from below, would you expect the P wave to be recorded? Which section of the fibre would you expect to be noisiest?

<details><summary>▸ Answers</summary>

Yes: the P-wave particle motion is approximately parallel to the fibre (see 6A.2 and 6A.3). The section at the surface (x < 0), in the conduit and recording building, is much noisier than the fibre cemented in the well.

</details>

In [ ]:
forge = du.load_das("forge_event.npz")
print(forge)
forge_bp = du.bandpass(forge.data, forge.fs, 10, 200)
in_well = slice(300, 1100)     # colour scale from channels in the well (the surface section is much noisier)
fig, axs = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
forge_raw = du.detrend(forge.data)       # remove each channel's mean and linear trend
du.plot_record(forge, forge_raw, ax=axs[0], vmax=np.percentile(abs(forge_raw[:, in_well]), 99), title="raw (detrended)")
du.plot_record(forge, forge_bp, ax=axs[1], vmax=np.percentile(abs(forge_bp[:, in_well]), 99), title="10–200 Hz band-pass")
plt.tight_layout()

### 6A.2 Denoising
These data are sampled at 2000 Hz (Rutford: 1000 Hz) and were recorded with a different interrogator in a different setting. Apply DAS-N2N directly to the raw data:

In [ ]:
with du.timer("DAS-N2N on FORGE"):
    forge_den = du.dasn2n_denoise(model, forge.data)
du.compare({"raw (detrended)": forge_raw, "DAS-N2N": forge_den}, forge, vmax=50, same_scale=False);

Much of the random noise has been removed, but horizontal stripes remain. These are common-mode noise: fluctuations that appear simultaneously on every channel, typically originating in the interrogator (for example, laser phase noise) rather than in the ground. At Rutford, such noise would be identical on both fibres, so it behaves like signal in Noise2Noise training: it is predictable from the input, and DAS-N2N preserves it.

Common-mode noise is usually removed before denoising. After removing each channel's mean, the median across a set of channels is computed at each time sample and subtracted from every channel. The channels used to estimate it should lie within the well and avoid the much noisier surface section; here we use channels 400–1100.

**Q6A.2** Why might including the surface channels (x < 0) in this estimate make the result worse rather than better?

In [ ]:
cm_channels = slice(400, 1100)          # channel indices (not metres) used to estimate the common-mode noise
forge_cm = du.remove_common_mode(forge.data, cm_channels)
with du.timer("DAS-N2N after common-mode removal"):
    forge_cm_den = du.dasn2n_denoise(model, forge_cm)
du.compare({"DAS-N2N on raw data": forge_den, "common-mode removal, then DAS-N2N": forge_cm_den},
           forge, vmax=50, tlim=(0.75, 1.75), xlim=(-50, 1050));

<details><summary>▸ Answers</summary>

The common-mode estimate should represent noise that is identical on every channel. The surface channels have amplitudes up to two orders of magnitude larger, dominated by their own local noise, so including them would contaminate the estimate and subtracting it would add noise to the channels in the well. The median is used rather than the mean because it is insensitive to a small number of anomalous channels (e.g. the glitch at ≈800 m and ≈0.3 s). After removal, the horizontal stripes are absent from the DAS-N2N output; the vertical stripes that remain are individual noisy channels.

</details>

### 6A.3 P-wave velocity along the well
✏️ Measure the apparent P-wave velocity along the fibre, as in Part 5, once in the deeper part of the well (≈600–1000 m) and once in the shallower part (≈100–400 m).

In [ ]:
#@title 🎚️ Zoomed view { run: "auto" }
t_start = 0.85   #@param {type:"slider", min:0, max:2.4, step:0.05}
t_end = 1.25     #@param {type:"slider", min:0.1, max:2.5, step:0.05}
ax = du.plot_record(forge, forge_cm_den, vmax=40, tlim=(t_start, t_end), xlim=(0, 1000))
ax.grid(alpha=0.5); ax.minorticks_on(); ax.grid(which="minor", alpha=0.2)

In [ ]:
# ✏️ P-wave picks as (depth in m, time in s), two in each part of the well
deep_1, deep_2 = None, None             # e.g. (1000, 0.89), (600, ...)
shallow_1, shallow_2 = None, None       # e.g. (400, ...), (100, ...)

du.require(deep_1, deep_2, shallow_1, shallow_2)
v_deep = abs(deep_1[0] - deep_2[0]) / abs(deep_1[1] - deep_2[1])
v_shallow = abs(shallow_1[0] - shallow_2[0]) / abs(shallow_1[1] - shallow_2[1])
print(f"apparent P velocity: deep ≈ {v_deep:.0f} m/s, shallow ≈ {v_shallow:.0f} m/s")

**Q6A.3**
* Why is the P wave clearly recorded here but not at Rutford?
* Why does the P-wave arrival curve as it approaches the surface, and what does this indicate about the subsurface?
* A second, larger arrival is recorded later at the bottom of the well (≈1.08 s at 1000 m). What might it be?
* Why might DAS-N2N perform well on these data despite the differences in sampling rate, instrument and setting?

<details><summary>▸ Answers</summary>

P picks, for example: `deep_1, deep_2 = (1000, 0.893), (600, 0.975)` and `shallow_1, shallow_2 = (400, 1.035), (100, 1.165)`.

* The fibre is approximately parallel to the ray path and to the P-wave particle motion, which is the orientation of maximum sensitivity, in contrast to the horizontal fibre at Rutford.
* Apparent velocity ≈4.9 km/s at depth and ≈2.3 km/s near the surface: seismic velocity increases with depth, so the wave slows as it approaches the surface. Apparent velocities are upper bounds on the true velocities if the source is laterally offset from the well.
* Most probably the S wave, which is slower and of larger amplitude than the P wave; this can be left as an open question.
* The network operates on normalised 128 × 96 patches and primarily learns to distinguish locally coherent structure from incoherent noise, which depends relatively little on the absolute sampling rate or units, provided the noise is incoherent.

</details>

---
## Part 6B. Whale calls on a seafloor telecommunications cable

The Ocean Observatories Initiative (OOI) Regional Cabled Array includes two fibre-optic cables that run offshore from Pacific City, Oregon. During a maintenance period in November 2021, spare fibres were used for a four-day DAS experiment (Wilcock et al., 2023). This file contains 15 s of data from the south cable, between 28 and 52 km from the shore station: about 12,000 channels at 2 m spacing, sampled at 200 Hz.

Signals reported in these data by Wilcock et al. (2023) include:

| Signal | Frequency band | Characteristics |
|---|---|---|
| ocean waves and microseisms | below ~2 Hz | dominate the unfiltered data |
| fin whale calls | ~15–27 Hz | ~1 s calls; frequency decreases during each call |
| blue whale A calls | ~10–18 Hz | V-shaped moveout, with the apex at the point on the cable closest to the whale |
| poorly coupled cable sections | broad band | vertical stripes |

The speed of sound in seawater is about 1480 m/s.

In [ ]:
ooi = du.load_das("ooi_whales.npz")
print(ooi)
print(f"{ooi.data.nbytes / 1e6:.0f} MB in memory for 15 s of data")
du.plot_record(ooi, perc=98);

**Q6B.1** The unfiltered data are dominated by slowly varying, low-frequency signals. Use the spectrogram below (with `distance_km` set to various positions) to estimate their frequency range.

In [ ]:
#@title 🎚️ Spectrogram averaged over 100 channels { run: "auto" }
distance_km = 38.6   #@param {type:"slider", min:28.2, max:51.8, step:0.2}
from scipy import signal as sps
c = int((distance_km * 1000 - ooi.x0) / ooi.dx)
tr = ooi.data[:, max(0, c - 50): c + 50]
tr = tr - tr.mean(axis=0)
f, t, S = sps.spectrogram(tr, fs=ooi.fs, nperseg=128, noverlap=120, axis=0)
S = 10 * np.log10(S.mean(axis=1))
plt.figure(figsize=(10, 4))
plt.pcolormesh(t, f, S, shading="auto", vmin=np.percentile(S, 5), vmax=np.percentile(S, 99.5))
plt.ylim(0, 60); plt.xlabel("time (s)"); plt.ylabel("frequency (Hz)"); plt.colorbar(label="dB")
plt.title(f"spectrogram, 100 channels centred on {distance_km} km");

**Q6B.2** At 38.6 km, a narrow-band signal persists for most of the record. At what frequency?

**Q6B.3** Move to about 46.6 km. Can you identify a call of roughly one second whose frequency decreases over time? At what time and over what frequency range?

🎚️ Now band-pass the full section. Try 10.5–18 Hz and 15–27 Hz.

In [ ]:
#@title 🎚️ Band-pass filter { run: "auto" }
band_min = 15   #@param {type:"slider", min:0.5, max:60, step:0.5}
band_max = 27   #@param {type:"slider", min:1, max:99, step:1}
ooi_bp = du.bandpass(ooi.data, ooi.fs, band_min, band_max)
du.plot_record(ooi, ooi_bp, perc=99, title=f"{band_min}–{band_max} Hz");

**Q6B.4** In the 10.5–18 Hz band, where is the apex of the repeating V-shaped arrivals, and what does its position indicate?

**Q6B.5** In the 15–27 Hz band, two long, nearly straight arrivals cross the section. ✏️ Pick two points on one of them and compute its apparent velocity along the cable. A sound wave travelling along the cable has an apparent velocity equal to the sound speed $c$; one arriving at an angle $\phi$ to the cable has apparent velocity $c/\cos\phi$.

In [ ]:
pick_1 = None   # ✏️ (distance in km, time in s)
pick_2 = None   # ✏️

du.require(pick_1, pick_2)
v_app = abs(pick_1[0] - pick_2[0]) * 1000 / abs(pick_1[1] - pick_2[1])
phi = np.degrees(np.arccos(min(1, 1480 / v_app)))
print(f"apparent velocity ≈ {v_app:.0f} m/s, i.e. arrival at ≈ {phi:.0f}° to the cable")

**Q6B.6** Apply DAS-N2N to a 6 km section of the cable. As in Lapins et al. (2024), the data are first high-pass filtered at 10 Hz to remove the large low-frequency ocean signal. (On a CPU this takes about 10 s; the full section takes about a minute.) DAS-N2N was trained on data with 1 m channel spacing, sampled at 1000 Hz, from an ice stream. How well does it perform here?

In [ ]:
my_km = (34, 40)    # section of cable to denoise (km); try others

c1, c2 = [int((km * 1000 - ooi.x0) / ooi.dx) for km in my_km]
sub = du.DASRecord(ooi.data[:, c1:c2], ooi.fs, ooi.dx, ooi.x[c1], ooi.starttime, f"OOI {my_km[0]}–{my_km[1]} km")
sub_hp = du.bandpass(sub.data, sub.fs, 10, 99)
with du.timer("DAS-N2N"):
    sub_den = du.dasn2n_denoise(model, sub_hp)
du.compare({"10–99 Hz": sub_hp, "10–99 Hz + DAS-N2N": sub_den,
            f"{band_min}–{band_max} Hz": du.bandpass(sub.data, sub.fs, band_min, band_max)},
           sub, same_scale=False);

<details><summary>▸ Answers</summary>

This file is the one shown in Fig. 3a of Wilcock et al. (2023).
* Q6B.1: the unfiltered data are dominated by energy below ~2 Hz (primary microseisms in shallow water and secondary microseisms in deeper water, according to Wilcock et al.).
* Q6B.2: ≈14 Hz.
* Q6B.3: a downsweep from ≈25 to ≈18 Hz at ≈12–13 s, characteristic of a fin whale call.
* Q6B.4: the apex is at ≈38.6 km, the point on the cable closest to the calling blue whale (Wilcock et al. report the closest point at 38 km).
* Q6B.5: for example `pick_1 = (40.0, 9.9)`, `pick_2 = (49.9, 13.75)`. Fin whale arrivals at ≈44–50 km (0–3 s) and ≈39–50 km (≈9.5–13.8 s). Apparent velocities are ≈2.4–2.7 km/s, i.e. φ ≈ 50–55°. The nearly linear moveout over ~10 km indicates a distant source (Wilcock et al. locate fin whales 25–75 km offshore).
* Q6B.6: after high-pass filtering, DAS-N2N performs well despite the different channel spacing, sampling rate and setting.

</details>

---
## Part 7. Optional: training a Noise2Noise model (GPU recommended)

The file `rutford_fibre_pairs_10s.npz` contains 10 s of data from both Rutford fibres. The cells below train an untrained copy of the DAS-N2N network on two kinds of input–target pairs:

1. the same channel on fibre 1 and fibre 2 (Noise2Noise pairs, as used for DAS-N2N);
2. neighbouring channels (`c` and `c + 1`) on fibre 1, whose noise was shown in 4.1 to be strongly correlated;

and compare the results with the pre-trained model on event 2. Each model takes about 10 s to train on a GPU and about 5 min on a CPU.

In [ ]:
import torch
from dasn2n import DASN2N

device = "cuda" if torch.cuda.is_available() else "cpu"
print("training on", device)
pairs = du.load_das("rutford_fibre_pairs_10s.npz")

def standardise(a):
    a = a - a.mean(axis=0, keepdims=True)
    return (a / a.std(axis=0, keepdims=True)).astype(np.float32)

fibre1, fibre2 = standardise(pairs.data), standardise(pairs.fibre2)
rng = np.random.default_rng(0)

def random_patches(inp, tgt, n=24, channel_shift=0):
    """n random 128x96 patches from `inp`, and the corresponding patches from `tgt` (optionally offset by some channels)."""
    i = rng.integers(0, inp.shape[0] - 128, n)
    j = rng.integers(0, inp.shape[1] - 96 - channel_shift, n)
    a = np.stack([inp[ii:ii + 128, jj:jj + 96] for ii, jj in zip(i, j)])
    b = np.stack([tgt[ii:ii + 128, jj + channel_shift:jj + channel_shift + 96] for ii, jj in zip(i, j)])
    flip = rng.random(n) < 0.5                           # data augmentation: random flips along the channel axis
    a[flip], b[flip] = a[flip][:, :, ::-1], b[flip][:, :, ::-1]
    return torch.from_numpy(a.copy()).to(device), torch.from_numpy(b.copy()).to(device)

def train(pair_type, n_steps=600, lr=1e-3):
    net = DASN2N().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    losses = []
    for step in range(n_steps):
        if pair_type == "fibres":
            x, y = random_patches(fibre1, fibre2)
        else:
            x, y = random_patches(fibre1, fibre1, channel_shift=1)
        loss = torch.mean((net(x) - y) ** 2)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return net.eval(), losses

In [ ]:
with du.timer("training on fibre pairs"):
    net_fibres, loss_fibres = train("fibres")
with du.timer("training on neighbouring channels"):
    net_neigh, loss_neigh = train("neighbours")

plt.figure(figsize=(8, 3))
plt.plot(np.convolve(loss_fibres, np.ones(50) / 50, "valid"), label="fibre pairs")
plt.plot(np.convolve(loss_neigh, np.ones(50) / 50, "valid"), label="neighbouring channels")
plt.xlabel("step"); plt.ylabel("training loss (MSE)"); plt.legend();

**Q7** Which model reaches the lower training loss? Does a lower training loss indicate a better denoiser in this case?

In [ ]:
du.compare({"event 2 raw": rec2.data,
            "pre-trained DAS-N2N": den2,
            "trained on fibre pairs": du.dasn2n_denoise(net_fibres, rec2.data),
            "trained on neighbouring channels": du.dasn2n_denoise(net_neigh, rec2.data)},
           rec2, vmax=50, tlim=(0.8, 2.4));

<details><summary>▸ Answers</summary>

The model trained on neighbouring channels reaches a much lower loss (≈0.005): because the noise on neighbouring channels is strongly correlated, the loss is minimised by approximately reproducing the input, and its output is essentially the noisy data. The loss of the fibre-pair model remains close to 1, because the target noise is unpredictable and accounts for almost all of the variance, yet after ~600 steps its output is comparable to that of the published model (trained for 30 epochs on 2 hours of data). In self-supervised learning, the value of the training loss alone does not indicate whether the model has learned the intended mapping.

</details>

---
## Summary questions
1. What does DAS provide that a network of conventional seismometers does not, and vice versa?
2. Which kinds of noise does DAS-N2N suppress effectively, which does it not, and why?
3. In your own research area, where might a fibre-optic cable (an existing telecommunications fibre, a borehole, a glacier, an urban network, a volcano) provide useful observations? Which signals would be of interest, and which noise sources would be a concern?

### References
* Hudson, T. S., et al. (2021). Distributed Acoustic Sensing (DAS) for natural microseismicity studies: A case study from Antarctica. *Journal of Geophysical Research: Solid Earth*, 126, e2020JB021493. https://doi.org/10.1029/2020JB021493
* Lapins, S., et al. (2024). DAS-N2N: machine learning distributed acoustic sensing (DAS) signal denoising without clean data. *Geophysical Journal International*. https://doi.org/10.1093/gji/ggad460
* Lehtinen, J., et al. (2018). Noise2Noise: learning image restoration without clean data. *Proceedings of the 35th International Conference on Machine Learning*. https://arxiv.org/abs/1803.04189
* Smith, A. M. (1997). Basal conditions on Rutford Ice Stream, West Antarctica, from seismic observations. *Journal of Geophysical Research*.
* Walter, F., et al. (2020). Distributed acoustic sensing of microseismic sources and wave propagation in glaciated terrain. *Nature Communications*.
* Wilcock, W. S. D., Abadi, S., & Lipovsky, B. P. (2023). Distributed acoustic sensing recordings of low-frequency whale calls and ship noise offshore Central Oregon. https://doi.org/10.1121/10.0017104

### Data
Rutford Ice Stream DAS: BEAMISH project (Lapins et al., 2024; Hudson et al., 2021); geophones: EarthScope network 6L. Utah FORGE DAS: Utah FORGE / US DOE Geothermal Data Repository. OOI DAS: OOI Regional Cabled Array community experiment (Wilcock et al., 2023).